[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Deep_Learning_From_Scratch.ipynb)

# Deep Learning From Scratch — Every Gradient by Hand

**Companion notebook to** [`playbooks/ai/deep-learning/02_Concepts.md`](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/deep-learning/02_Concepts.md).

This notebook trains a tiny neural network on three houses **using only NumPy** — no PyTorch, no TensorFlow, no autograd. Every weight, every gradient, every update is computed by hand.

**Why this matters.** When you write `loss.backward()` in PyTorch, the framework runs exactly what is in this notebook — just with bigger matrices and more layers. If the math here makes sense, the math at scale makes sense.

## What you will see
1. Build a 2-hidden-neuron network from scratch
2. Train on House A (Cycle 1) — every gradient computed manually
3. Train on House B (Cycle 2) — same pattern, updated weights
4. Predict on a new house — see how rough the model is after only 2 steps
5. Train *properly* — loop over all houses for many epochs, watch the loss curve
6. **Verify** — run the same example through PyTorch autograd and confirm every gradient matches

By the end, `loss.backward()` will stop being magic.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
print("NumPy version:", np.__version__)

## 1. The Dataset — Three Houses

We want to predict house price from square footage. Three training examples.

| House | Sqft | Price |
|---|---:|---:|
| A | 1,000 | \$200,000 |
| B | 2,000 | \$350,000 |
| C | 3,000 | \$450,000 |

### Why we scale

Real numbers like 1,500 sqft and \$350,000 live on different scales. Feed those raw into a neural network and the gradients explode, the loss goes to NaN, training fails on batch one.

**Scaling rule:** divide every input and output by a constant so they live near 1.

```
x = sqft  / 1000
y = price / 100000
```

In [ ]:
# Raw data
houses = [
    ("A", 1000, 200_000),
    ("B", 2000, 350_000),
    ("C", 3000, 450_000),
]

# Scale: keep everything near 1 so gradients behave
X = np.array([sqft / 1000 for _, sqft, _ in houses])
Y = np.array([price / 100_000 for _, _, price in houses])

print("Scaled inputs  x:", X)
print("Scaled targets y:", Y)

## 2. The Network — Two Hidden Neurons

```
Input x
   ↓
Hidden Layer (2 neurons with ReLU)
   ↓
Output Layer (1 neuron, no activation — regression)
   ↓
y_pred
```

The forward pass:

```
z1 = w1*x + b1            h1 = ReLU(z1)
z2 = w2*x + b2            h2 = ReLU(z2)
y_pred = v1*h1 + v2*h2 + c
```

Loss (MSE with a 0.5 convenience factor):

```
L = 0.5 * (y_pred - y)**2
```

Seven learnable parameters: `w1, b1, w2, b2, v1, v2, c`.

In [ ]:
def relu(z):
    return max(0.0, z)

def relu_deriv(z):
    return 1.0 if z > 0 else 0.0

def forward(x, w1, b1, w2, b2, v1, v2, c):
    z1 = w1 * x + b1
    h1 = relu(z1)
    z2 = w2 * x + b2
    h2 = relu(z2)
    y_pred = v1 * h1 + v2 * h2 + c
    # Return everything so the backward pass can use the cached values
    return {"z1": z1, "h1": h1, "z2": z2, "h2": h2, "y_pred": y_pred}

def loss_fn(y_pred, y):
    return 0.5 * (y_pred - y) ** 2

print("Network defined.")

## 3. Initial Values

In real training, weights start as small random numbers. For a clean walkthrough we pick simple values.

In [ ]:
# Starting weights and biases
w1 = 0.5
b1 = 0.1
w2 = -0.4
b2 = 1.0

# Output weights and bias
v1 = 0.3
v2 = 0.2
c  = 0.5

# Learning rate
alpha = 0.1

print(f"w1={w1}, b1={b1}, w2={w2}, b2={b2}")
print(f"v1={v1}, v2={v2}, c={c}, alpha={alpha}")

## 4. Cycle 1 — Train on House A (x=1.0, y=2.0)

We will:
1. Run the forward pass
2. Compute the loss
3. Compute every gradient by hand using the chain rule
4. Update every weight

Then verify our prediction got closer to the truth.

### 4.1 Forward Pass

In [ ]:
x = X[0]      # House A: x = 1.0
y = Y[0]      # House A: y = 2.0

state = forward(x, w1, b1, w2, b2, v1, v2, c)
print(f"z1 = {w1}*{x} + {b1} = {state['z1']}")
print(f"h1 = ReLU({state['z1']}) = {state['h1']}")
print(f"z2 = {w2}*{x} + {b2} = {state['z2']}")
print(f"h2 = ReLU({state['z2']}) = {state['h2']}")
print(f"y_pred = {v1}*{state['h1']} + {v2}*{state['h2']} + {c} = {state['y_pred']}")
print()
print(f"Model predicts: ${state['y_pred'] * 100_000:,.0f}")
print(f"Actual price:    ${y * 100_000:,.0f}")

### 4.2 Loss

In [ ]:
L = loss_fn(state['y_pred'], y)
print(f"L = 0.5 * ({state['y_pred']} - {y})**2 = {L}")

### 4.3 Backward Pass — All Gradients by Hand

The seed is `dL/dy_pred`. Since `L = 0.5*(y_pred - y)**2`:

```
dL/dy_pred = y_pred - y
```

This single number — the **error signal** — drives every gradient downstream.

In [ ]:
# Error signal — the seed of every gradient
dL_dy_pred = state['y_pred'] - y
print(f"dL/dy_pred = {state['y_pred']} - {y} = {dL_dy_pred}")
print()

# Output layer gradients (y_pred is a direct linear combination of h1, h2, c)
dL_dv1 = dL_dy_pred * state['h1']
dL_dv2 = dL_dy_pred * state['h2']
dL_dc  = dL_dy_pred

print(f"dL/dv1 = {dL_dy_pred} * {state['h1']} = {dL_dv1}")
print(f"dL/dv2 = {dL_dy_pred} * {state['h2']} = {dL_dv2}")
print(f"dL/dc  = {dL_dc}")

### 4.4 Hidden Layer Gradients — The Chain Rule

For `w1`, the dependency chain is:

```
w1  →  z1  →  h1  →  y_pred  →  L
```

Walk it backward:

```
dL/dw1 = (dL/dy_pred) * (dy_pred/dh1) * (dh1/dz1) * (dz1/dw1)
       = (y_pred - y) *  v1            *  ReLU'   *  x
```

Plain English:

> dL/dw1 = (how wrong we are) × (how much this neuron matters) × (whether the neuron is firing) × (how much input affects the neuron)

In [ ]:
# Chain rule for each hidden parameter
relu1 = relu_deriv(state['z1'])   # 1 if z1 > 0, else 0
relu2 = relu_deriv(state['z2'])

dL_dw1 = dL_dy_pred * v1 * relu1 * x
dL_db1 = dL_dy_pred * v1 * relu1 * 1
dL_dw2 = dL_dy_pred * v2 * relu2 * x
dL_db2 = dL_dy_pred * v2 * relu2 * 1

print(f"dL/dw1 = {dL_dy_pred} * {v1} * {relu1} * {x} = {dL_dw1}")
print(f"dL/db1 = {dL_dy_pred} * {v1} * {relu1} * 1 = {dL_db1}")
print(f"dL/dw2 = {dL_dy_pred} * {v2} * {relu2} * {x} = {dL_dw2}")
print(f"dL/db2 = {dL_dy_pred} * {v2} * {relu2} * 1 = {dL_db2}")

### 4.5 Update Every Weight

Update rule:

```
new = old - learning_rate * gradient
```

In [ ]:
w1 = w1 - alpha * dL_dw1
b1 = b1 - alpha * dL_db1
w2 = w2 - alpha * dL_dw2
b2 = b2 - alpha * dL_db2
v1 = v1 - alpha * dL_dv1
v2 = v2 - alpha * dL_dv2
c  = c  - alpha * dL_dc

print("Weights after Cycle 1 (House A):")
print(f"  w1 = {w1:.4f}    b1 = {b1:.4f}")
print(f"  w2 = {w2:.4f}    b2 = {b2:.4f}")
print(f"  v1 = {v1:.4f}    v2 = {v2:.4f}")
print(f"  c  = {c:.4f}")

## 5. Cycle 2 — Train on House B (x=2.0, y=3.5)

Same exact pattern, with the updated weights from Cycle 1. Let's verify the prediction changes.

In [ ]:
x = X[1]      # House B: x = 2.0
y = Y[1]      # House B: y = 3.5

state = forward(x, w1, b1, w2, b2, v1, v2, c)
L = loss_fn(state['y_pred'], y)

print(f"y_pred = {state['y_pred']:.4f}  (predicted ${state['y_pred'] * 100_000:,.0f})")
print(f"actual = {y}        (actual    ${y * 100_000:,.0f})")
print(f"L      = {L:.4f}")
print()

# Same chain rule, applied with current weights
dL_dy_pred = state['y_pred'] - y
relu1 = relu_deriv(state['z1'])
relu2 = relu_deriv(state['z2'])

dL_dv1 = dL_dy_pred * state['h1']
dL_dv2 = dL_dy_pred * state['h2']
dL_dc  = dL_dy_pred
dL_dw1 = dL_dy_pred * v1 * relu1 * x
dL_db1 = dL_dy_pred * v1 * relu1 * 1
dL_dw2 = dL_dy_pred * v2 * relu2 * x
dL_db2 = dL_dy_pred * v2 * relu2 * 1

w1 -= alpha * dL_dw1
b1 -= alpha * dL_db1
w2 -= alpha * dL_dw2
b2 -= alpha * dL_db2
v1 -= alpha * dL_dv1
v2 -= alpha * dL_dv2
c  -= alpha * dL_dc

print("Weights after Cycle 2:")
print(f"  w1 = {w1:.4f}    b1 = {b1:.4f}")
print(f"  w2 = {w2:.4f}    b2 = {b2:.4f}")
print(f"  v1 = {v1:.4f}    v2 = {v2:.4f}")
print(f"  c  = {c:.4f}")

## 6. Predict for a New House (sqft = 2500)

The model has only seen two training steps. The prediction will be poor — the point is to see that the math runs end-to-end.

In [ ]:
x_new = 2.5     # 2500 sqft -> x = 2.5
state = forward(x_new, w1, b1, w2, b2, v1, v2, c)

predicted_price = state['y_pred'] * 100_000
print(f"For sqft=2500:")
print(f"  y_pred = {state['y_pred']:.4f}")
print(f"  predicted price ≈ ${predicted_price:,.0f}")
print()
print("(Real models train for thousands of steps. We did 2.)")

## 7. Train Properly — Many Epochs

Now run the same loop over **all three houses** for **500 epochs** (an epoch = one full pass through the training set). Watch the loss curve drop.

This is what a real training loop looks like — same five steps, just iterated.

In [ ]:
# Reset to initial weights for a fair comparison
w1, b1 = 0.5, 0.1
w2, b2 = -0.4, 1.0
v1, v2, c = 0.3, 0.2, 0.5
alpha = 0.05   # smaller learning rate is more stable for many epochs

losses = []
EPOCHS = 500

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for x, y in zip(X, Y):
        # 1. Forward
        state = forward(x, w1, b1, w2, b2, v1, v2, c)

        # 2. Loss
        epoch_loss += loss_fn(state['y_pred'], y)

        # 3. Backward (the same chain rule from Cycle 1)
        dL_dy_pred = state['y_pred'] - y
        relu1 = relu_deriv(state['z1'])
        relu2 = relu_deriv(state['z2'])

        dL_dv1 = dL_dy_pred * state['h1']
        dL_dv2 = dL_dy_pred * state['h2']
        dL_dc  = dL_dy_pred
        dL_dw1 = dL_dy_pred * v1 * relu1 * x
        dL_db1 = dL_dy_pred * v1 * relu1
        dL_dw2 = dL_dy_pred * v2 * relu2 * x
        dL_db2 = dL_dy_pred * v2 * relu2

        # 4. Update
        w1 -= alpha * dL_dw1; b1 -= alpha * dL_db1
        w2 -= alpha * dL_dw2; b2 -= alpha * dL_db2
        v1 -= alpha * dL_dv1; v2 -= alpha * dL_dv2
        c  -= alpha * dL_dc

    losses.append(epoch_loss / len(X))

print(f"Loss at epoch 1:    {losses[0]:.4f}")
print(f"Loss at epoch 100:  {losses[99]:.4f}")
print(f"Loss at epoch 500:  {losses[-1]:.4f}")

In [ ]:
# Plot the loss curve — the heartbeat of training
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Average loss across the 3 houses")
plt.title("Loss curve — going down means the model is learning")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Predict for all training houses and a held-out 2500 sqft house
print("After 500 epochs:")
print()
print(f"{'House':>6}  {'Sqft':>6}  {'Predicted':>12}  {'Actual':>10}")
print("-" * 42)
for (label, sqft, price), x, y in zip(houses, X, Y):
    state = forward(x, w1, b1, w2, b2, v1, v2, c)
    pred = state['y_pred'] * 100_000
    print(f"{label:>6}  {sqft:>6}  ${pred:>11,.0f}  ${price:>9,.0f}")

# Held-out
state = forward(2.5, w1, b1, w2, b2, v1, v2, c)
print(f"{'?':>6}  {2500:>6}  ${state['y_pred']*100_000:>11,.0f}  {'(held-out)':>10}")

## 8. Verify — Same Math, PyTorch Autograd

The whole point of `loss.backward()` is that PyTorch computes exactly what we computed by hand. Let's prove it.

We will:
1. Reset to the original initial weights
2. Run **one** forward+backward pass on House A in PyTorch
3. Print the gradients PyTorch computes
4. Compare to our manual numbers from Cycle 1

If they match, you have just verified that the chain rule we derived **is** what PyTorch runs.

In [ ]:
import torch

# Same initial values, this time as torch tensors with requires_grad=True
W1 = torch.tensor(0.5,  requires_grad=True)
B1 = torch.tensor(0.1,  requires_grad=True)
W2 = torch.tensor(-0.4, requires_grad=True)
B2 = torch.tensor(1.0,  requires_grad=True)
V1 = torch.tensor(0.3,  requires_grad=True)
V2 = torch.tensor(0.2,  requires_grad=True)
C  = torch.tensor(0.5,  requires_grad=True)

x = torch.tensor(1.0)   # House A
y = torch.tensor(2.0)

# Forward pass — same equations
Z1 = W1 * x + B1
H1 = torch.relu(Z1)
Z2 = W2 * x + B2
H2 = torch.relu(Z2)
Y_PRED = V1 * H1 + V2 * H2 + C
LOSS = 0.5 * (Y_PRED - y) ** 2

# Backward pass — autograd computes every gradient
LOSS.backward()

print("PyTorch autograd gradients (should match our manual numbers):")
print(f"  dL/dw1 = {W1.grad.item():+.4f}    (manual: -0.3600)")
print(f"  dL/db1 = {B1.grad.item():+.4f}    (manual: -0.3600)")
print(f"  dL/dw2 = {W2.grad.item():+.4f}    (manual: -0.2400)")
print(f"  dL/db2 = {B2.grad.item():+.4f}    (manual: -0.2400)")
print(f"  dL/dv1 = {V1.grad.item():+.4f}    (manual: -0.7200)")
print(f"  dL/dv2 = {V2.grad.item():+.4f}    (manual: -0.7200)")
print(f"  dL/dc  = {C.grad.item():+.4f}    (manual: -1.2000)")

## What you just did

| Step | What you proved |
|---|---|
| Cycles 1–2 | You can train a neural network without any framework. The math is simple and explicit. |
| 500-epoch training | The same five steps, run in a loop, reduce the loss steadily. This is *all* training is. |
| PyTorch verification | Autograd computes the exact gradients you derived by hand. `loss.backward()` is not magic — it is automated chain rule. |

**Next:** [03 — Hello World](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/deep-learning/03_Hello_World.md) — build the same network in 30 lines of PyTorch.

**Or jump to architecture deep dives:** [CNN](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/deep-learning/architectures/cnn.md), [Transformer playbook](https://github.com/sunilmogadati/systems-in-production/tree/main/playbooks/ai/transformers), [GAN](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/deep-learning/architectures/gan.md), [U-Net](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/deep-learning/architectures/u-net.md).